In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Customers Profiling").getOrCreate()

c:\Users\LENOVO\Desktop\AdvancedPySparkPipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
customers = spark.read.parquet("../data/bronze/customers")

In [3]:
customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [4]:
customers.count()

99441

In [5]:
customers.show(10, truncate=False)

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|f2a1d75b74d9ec748af88e894cd87597|15ee900ec703c9a10082b12bcbdcb307|68590                   |jacunda              |PA            |
|f15272fe9d0e2ae3297185f18d3bac46|11e74a9cbe1158d1c934d4c166f59ff3|15056                   |sao jose do rio preto|SP            |
|7324ecb0ff143f561193d22bea7d63fb|c6be127fa6e30c6f705a205236a1f310|13302                   |itu                  |SP            |
|7accf3d920f47c07f5bfbc88f53f9926|a7f1a6dc9ba06844bd84b810efa56d33|45638                   |coaraci              |BA            |
|3680a273ddb333253fa2edc7d3f8a3f2|6cbfcc29787035834a40a46f5898436a|29700                  

In [ ]:
#null counts
from pyspark.sql.functions import col, when, count
customers.select([
    count(when(col(c).isNull(),c)).alias(c)
    for c in customers.columns
]).show(vertical=True)

-RECORD 0-----------------------
 customer_id              | 0   
 customer_unique_id       | 0   
 customer_zip_code_prefix | 0   
 customer_city            | 0   
 customer_state           | 0   



In [8]:
#duplicates
duplicate_customers=customers.groupBy("customer_id").agg(
    count("customer_id").alias("count")
).filter(col("count")>1)

In [9]:
duplicate_customers.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [10]:
#check unique states or cities
customers.select("customer_state").distinct().show()

+--------------+
|customer_state|
+--------------+
|            SC|
|            RO|
|            PI|
|            AM|
|            RR|
|            GO|
|            TO|
|            MT|
|            SP|
|            ES|
|            PB|
|            RS|
|            MS|
|            AL|
|            MG|
|            PA|
|            BA|
|            SE|
|            PE|
|            CE|
+--------------+
only showing top 20 rows


In [11]:
customers.select("customer_city").distinct().show(10, truncate=False)

+----------------------+
|customer_city         |
+----------------------+
|camacari              |
|itaberaba             |
|igrejinha             |
|iepe                  |
|barracao              |
|arapiraca             |
|pote                  |
|divino de sao lourenco|
|aguas de sao pedro    |
|jijoca de jericoacoara|
+----------------------+
only showing top 10 rows


In [12]:
#check zip code format
customers.select("customer_zip_code_prefix").show(10)

+------------------------+
|customer_zip_code_prefix|
+------------------------+
|                   68590|
|                   15056|
|                   13302|
|                   45638|
|                   29700|
|                   18055|
|                   44054|
|                   09812|
|                   70673|
|                   22793|
+------------------------+
only showing top 10 rows


In [13]:
customers.show(20, truncate=False)

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|f2a1d75b74d9ec748af88e894cd87597|15ee900ec703c9a10082b12bcbdcb307|68590                   |jacunda              |PA            |
|f15272fe9d0e2ae3297185f18d3bac46|11e74a9cbe1158d1c934d4c166f59ff3|15056                   |sao jose do rio preto|SP            |
|7324ecb0ff143f561193d22bea7d63fb|c6be127fa6e30c6f705a205236a1f310|13302                   |itu                  |SP            |
|7accf3d920f47c07f5bfbc88f53f9926|a7f1a6dc9ba06844bd84b810efa56d33|45638                   |coaraci              |BA            |
|3680a273ddb333253fa2edc7d3f8a3f2|6cbfcc29787035834a40a46f5898436a|29700                  